# Cost, speed and quality: choosing a version of the tutor

*For learning. Nothing here runs in CI, and nothing costs money. Some numbers are measured from the real
tutor, and the rest come from a **practice recording** (section 3). Every table says which it is.*

## What's going on, in one paragraph

The tutor in "Ask about this lesson" could run several ways. It could use a big model or a small one,
send its prompt as it does today or in a form the model can cache, or show a **prepared answer** written
ahead of time. Each version is a different mix of **quality** (are the answers good?), **speed** (how
long do learners wait?) and **cost** (what does each answer cost?). This notebook is the experiment that
picks one, the way a data scientist would. You write the rule first, measure all three scores on the
same questions, keep only the versions nothing beats, and then let the rule choose.

## Words you'll meet

| Word | Plain meaning |
|---|---|
| **Version** | One way of running the tutor: a model, a prompt layout, or prepared answers |
| **Token** | A piece of text, roughly four characters, that models read and write. Prices are per million tokens |
| **Prompt cache** | The model provider keeps the start of a prompt it has just read. If the next prompt starts the same way, that part is read from the cache at about a tenth of the price |
| **Cache hit rate** | The share of input tokens read from the cache |
| **Time to first words** | How long until the answer starts to appear. Often called *time to first token* (TTFT) |
| **p95** | The time that 95 out of 100 answers arrive within: the slow case learners notice |
| **Non-inferiority** | Showing a new version is *no worse* than the old one by more than an agreed margin |
| **Pareto front** | The versions no other version beats on every score at once |

In [ ]:
import math
import random
from collections import Counter

import ipytest
import matplotlib.pyplot as plt
import pandas as pd
import pytest

from drill import tradeoff as t
from drill import tutor_prepared as tp
from drill import tutor_prepared_eval as tpe
from drill.content.build import build_data

ipytest.autoconfig(clean="test_*")

DATA = build_data()          # every lesson, as the page sees it
CHIPS = tp.chips(DATA)       # the standard questions, as the page asks them
ANSWERS = tp.load()          # the prepared answers, each with its judge's verdict
SPLITS = tpe.load_splits()   # each lesson's frozen split: dev, test or holdout

# The questions this experiment uses: every standard question on a dev lesson that has a prepared answer.
QUESTIONS = [{"lesson_id": lid, "key": c["key"], "chip": c, "prepared": ANSWERS[lid][c["key"]]}
             for lid, items in CHIPS.items() if SPLITS.get(lid) == "dev"
             for c in items if ANSWERS.get(lid, {}).get(c["key"], {}).get("q") == c["q"]]
print(f"{len(QUESTIONS)} dev questions across {len({q['lesson_id'] for q in QUESTIONS})} lessons")

**What that cell did.** It loaded every lesson, the standard questions ("When does this stop working?",
"Why not B?"...), the prepared answers and the frozen splits. Then it kept the **dev** questions, the
practice set you may look at as often as you like. Test and holdout stay untouched until the end
(section 10).

One limit to say out loud: these are the **standard** questions only. Learners also type their own, and
those can't be prepared ahead. A full experiment needs a second set of real typed questions, where the
prepared version isn't an option.

## 0. Write the plan before looking at results

> **In plain words.** Three scores never agree on a winner by themselves: the best answers usually cost
> the most. So decide *before* the numbers how you'll trade them off. Here the rule goes in an order:
> quality must hold, then speed must be good enough, then the cheapest wins (lesson "Write the plan
> before the runs").

In [ ]:
PLAN = {
    "question": "Which version of the tutor should answer the standard questions?",
    "versions": ["prepared, else Sonnet", "Opus", "Sonnet", "Sonnet, cacheable prompt", "Haiku", "Haiku, cacheable prompt", "a local model"],
    "baseline": "Sonnet with today's prompt: a stand-in for the live tutor (set it to the model your app really uses)",
    "quality": "pass rate under the same judge, on the same dev questions",
    "speed": "p95 time to first words",
    "cost": "dollars per 1,000 answers, with the prompt cache counted",
    "decision_rule": "1) quality: the lower end of the paired difference with the baseline is above -3 points; "
                     "2) speed: p95 time to first words is at most 5 seconds; 3) cost: the cheapest version left wins",
    "splits": "explore on dev; confirm the winner once on test; holdout stays for a bigger decision",
    "timing": "batches aim to finish within 24 hours (a target, not a promise); answers then judging is two in a row, so allow 2 days before deciding",
}
MARGIN, P95_LIMIT = 0.03, 5.0
for k, v in PLAN.items():
    print(f"{k:14} {v}")

## 1. What one answer costs

> **In plain words.** A model charges for what it **reads** (input tokens) and what it **writes** (output
> tokens), at different prices per million. Writing usually costs about five times more than reading
> (lesson "Tokens and cost"). The prompt cache adds two more kinds of input. A **cache read** is the start
> of the prompt read from the cache, at a small share of the input price: a twentieth on Sonnet 5.5 and
> Opus 5.5, a tenth on most other models. A **cache write** is the first time that start is stored, at a
> quarter more than normal.

Here is one real Ask request: today's prompt for one question, and the prepared answer to it, measured in
tokens (roughly, at four characters a token).

In [ ]:
one = QUESTIONS[0]
prompt = t.today_messages(DATA, one["lesson_id"], one["chip"])[0]["content"]
answer = one["prepared"]["a"]
print(f"lesson {one['lesson_id']}, question: {one['chip']['q']}")
print(f"prompt about {t.estimate_tokens(prompt):,} tokens, answer about {t.estimate_tokens(answer):,} tokens")
print("prices per million tokens (input, output):", {m.removeprefix('databricks-claude-'): p for m, p in tp.PRICES.items()})

**Exercise.** Write `answer_cost`. `input_tokens` is the normal input, the part that was neither read
from nor written to the cache. That's how Claude's usage block counts it. Look up the model's prices in
`tp.PRICES` (a model that isn't there, like a local one, costs 0). Then add the four parts and divide by
a million:

- `input_tokens` at the input price;
- `cache_read` at the input price times `t.cache_read_rate(model)` (0.05 on Sonnet and Opus 5.5, 0.1 on others);
- `cache_write` at the input price times `t.CACHE_WRITE` (1.25);
- `output_tokens` at the output price.

In [ ]:
def answer_cost(model, input_tokens, output_tokens, cache_read=0, cache_write=0):
    """Dollars for one answer, with the prompt cache counted."""
    # TODO: pin, pout = tp.PRICES.get(model, (0.0, 0.0)); read = t.cache_read_rate(model); add the four parts; divide by 1e6
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_plain_answer():
    assert answer_cost("databricks-claude-sonnet-5-5", 2000, 300) == pytest.approx((2000 * 2 + 300 * 10) / 1e6)


def test_cache_read_is_a_twentieth_on_sonnet_5_5():
    assert answer_cost("databricks-claude-sonnet-5-5", 200, 300, cache_read=1800) == pytest.approx((200 * 2 + 90 * 2 + 300 * 10) / 1e6)


def test_cache_read_is_a_tenth_on_haiku():
    assert answer_cost("databricks-claude-haiku-5-5", 200, 300, cache_read=1800) == pytest.approx((200 * 0.1 + 180 * 0.1 + 300 * 0.5) / 1e6)


def test_first_write_costs_more_than_no_cache():
    assert answer_cost("databricks-claude-sonnet-5-5", 200, 300, cache_write=1800) > answer_cost("databricks-claude-sonnet-5-5", 2000, 300)


def test_a_local_model_costs_nothing():
    assert answer_cost("a-local-model", 2000, 300) == 0.0

In [ ]:
for model in ("databricks-claude-opus-5-5", "databricks-claude-sonnet-5-5", "databricks-claude-haiku-5-5"):
    each = answer_cost(model, t.estimate_tokens(prompt), t.estimate_tokens(answer))
    print(f"{model.removeprefix('databricks-claude-'):10} ${each:.4f} an answer, ${1000 * each:.2f} for 1,000")

**What to look for.** The output is a smaller share of the tokens but a big share of the bill, because
writing costs five times more than reading. A version that writes shorter answers saves money even on the
same model. That's one reason the experiment measures real answer lengths, not a guess.

## 2. Could the prompt cache hit at all? (measured)

> **In plain words.** The cache only helps if the **start** of the prompt is the same from one request to
> the next, and long enough. Providers won't cache a start shorter than a minimum: **512 tokens** on
> the Claude 5.5 models, **1,024** on GPT models. It differs by model, so check the provider's
> prompt-caching page for the one you test. On Claude the request must also ask for caching (a
> `cache_control` mark); GPT models cache by themselves. And the cache forgets after a few minutes
> without use.

The tutor's prompt starts with the same text for every question on a lesson: the tutor's instructions,
then the lesson. The question comes next, then the rules. So the start that **could** be cached is the
instructions plus the lesson. The **cacheable** version (`t.cacheable_messages`) moves the rules up so
they join the fixed start, and marks it for the cache. The model sees the same words in a different order.

In [ ]:
starts = pd.DataFrame([{"lesson_id": lid, "today": t.fixed_start(DATA, lid, items[0], "today"),
                        "cacheable": t.fixed_start(DATA, lid, items[0], "cacheable")}
                       for lid, items in CHIPS.items()])
print(f"fixed start, in tokens, across {len(starts)} lessons (measured from the real prompts):")
print(starts[["today", "cacheable"]].describe(percentiles=[.5, .9]).loc[["min", "50%", "90%", "max"]].round(0))
for col in ("today", "cacheable"):
    for model in ("databricks-claude-sonnet-5-5", "gpt-5.4-mini"):
        m = t.min_cacheable(model)
        print(f"{col:9}: {(starts[col] >= m).sum():3} of {len(starts)} lessons reach {m:,} tokens, the minimum on {model.removeprefix('databricks-claude-')}")
print("\nand today's request doesn't ask Claude to cache, so today nothing is cached at all")

**What to look for.** This is a real finding about the tutor, and it depends on the model.

- **Claude 5.5 (512 tokens).** Most lessons are already long enough today. With the rules moved up,
  every lesson is. So **today nothing is cached only because the request doesn't ask**. Asking is a
  small change with a real saving.
- **GPT (1,024 tokens).** Only a handful of lessons are long enough today, and about a third with the
  rules moved up. The rest are too short to cache, whatever the order.

The lesson for any app: look up the minimum for *your* model. The same prompt can cache well on one
model and not at all on another. Section 6 shows what it's worth in dollars.

These counts use the four-characters rule. A real run reads the exact numbers from the model's usage
block (`cache_read_input_tokens` and `cache_creation_input_tokens` on Claude).

**Exercise.** Write `hit_rate`: the share of **all** input tokens that were read from the cache. Each row
has `input_tokens`, and may have `cache_read` and `cache_write` (missing means 0). All input is the three
added together. With no input at all, return 0.

In [ ]:
def hit_rate(rows):
    """Share of all input tokens that were read from the cache."""
    # TODO: add up cache_read, and input_tokens + cache_read + cache_write, over the rows (use r.get(name, 0))
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_one_write_then_one_read():
    rows = [{"input_tokens": 200, "cache_write": 1800}, {"input_tokens": 200, "cache_read": 1800}]
    assert hit_rate(rows) == pytest.approx(1800 / 4000)


def test_no_cache_fields_means_no_hits():
    assert hit_rate([{"input_tokens": 500}]) == 0.0


def test_nothing_sent():
    assert hit_rate([]) == 0.0

**When would it hit?** `t.cache_pass` replays a list of calls in order. A lesson's start is written on its
first call. Each later call within 5 minutes reads it. After a longer gap it's written again. Below, the
same dev questions are sent two ways: **grouped**, as a learner works through a lesson (a question about
every 40 seconds), and **spread out** over a day in random order. Each is tried with three minimums, to
show what decides the answer.

In [ ]:
def send_order(questions, grouped, seed=7):
    rng = random.Random(seed)
    if grouped:
        at, out, last = 0.0, [], None
        for q in questions:
            at += 40 if q["lesson_id"] == last else 600
            out.append({"lesson_id": q["lesson_id"], "key": q["key"], "at": at}); last = q["lesson_id"]
        return out
    return sorted(({"lesson_id": q["lesson_id"], "key": q["key"], "at": rng.uniform(0, 8 * 3600)} for q in questions), key=lambda c: c["at"])


start_of = dict(zip(starts.lesson_id, starts.cacheable))
total_of = {(q["lesson_id"], q["key"]): t.estimate_tokens(t.today_messages(DATA, q["lesson_id"], q["chip"])[0]["content"]) for q in QUESTIONS}
rows = []
for grouped in (True, False):
    order = send_order(QUESTIONS, grouped)
    sizes = [total_of[(c["lesson_id"], c["key"])] for c in order]
    for minimum in (512, 1024, 2048):
        cached = t.cache_pass(order, start_of, minimum=minimum)
        usage = [{"input_tokens": s - c["cache_read"] - c["cache_write"], **c} for s, c in zip(sizes, cached)]
        rows.append({"order": "grouped by lesson" if grouped else "spread over a day", "minimum": minimum, "hit_rate": round(hit_rate(usage), 3)})
cache_table = pd.DataFrame(rows).pivot(index="order", columns="minimum", values="hit_rate")
cache_table

**What to look for.** Two things decide the hit rate, and they're the two things to get right in any app.
**Length**: below the minimum, the hit rate is zero whatever you do. **Timing**: calls about the same
lesson must come close together. A learner working through a lesson does that. The same questions spread
over a day mostly miss.

A related trap: in a test run that sends every question in parallel, the cache only fills once the first
answer starts. Send one call first, then the rest.

## 3. The practice recording (made up, on real foundations)

> **In plain words.** Running seven versions live on a thousand questions costs real money (section 8
> estimates how much). So that you can learn the method for free, the cell below **simulates** one run.
> The made-up parts are clearly separate from the real ones. Swap in a real recording later and every
> cell after this one works the same.

| Part | Where it comes from |
|---|---|
| Questions, lessons, prompt lengths | **Real**: the dev questions and today's prompts |
| Prepared answers' quality and length | **Real**: each one's recorded judge verdict, and its text |
| Prices, cache rates, which calls hit the cache | **Real** rules (`tp.PRICES`, `t.cache_pass`), applied to the order below |
| Live models' pass rates, answer lengths and timings | **Made up** for practice: the numbers in `VERSIONS` |

The made-up quality isn't pure coin flips. Questions whose prepared answer failed are treated as
**harder** for every version, and some lessons are harder than others. Real versions behave like that:
they tend to fail on the same questions. That's why the comparison in section 5 pairs answers.

In [ ]:
SONNET, OPUS, HAIKU = "databricks-claude-sonnet-5-5", "databricks-claude-opus-5-5", "databricks-claude-haiku-5-5"
# name: (model, prompt layout, pass rate on an ordinary question, typical seconds to first words, words a second, answer length vs prepared)
VERSIONS = {  # MADE UP for practice, except "prepared, else Sonnet", whose prepared half is real
    "prepared, else Sonnet":   (SONNET, "today", None, None, None, None),
    "Opus":                    (OPUS, "today", .95, 2.4, 45, 1.00),
    "Sonnet":                  (SONNET, "today", .93, 1.5, 70, 0.95),
    "Sonnet, cacheable prompt": (SONNET, "cacheable", .93, 1.5, 70, 0.95),
    "Haiku":                   (HAIKU, "today", .86, 0.8, 120, 0.90),
    "Haiku, cacheable prompt": (HAIKU, "cacheable", .86, 0.8, 120, 0.90),
    "a local model":           ("local", "today", .74, 3.2, 20, 1.20),
}
BASELINE = "Sonnet"


def logit(p):
    return math.log(p / (1 - p))


def simulate(seed=7):
    rng = random.Random(seed)
    lesson_ease = {lid: rng.gauss(0, 0.6) for lid in sorted({q["lesson_id"] for q in QUESTIONS})}
    luck = {(q["lesson_id"], q["key"]): rng.random() for q in QUESTIONS}      # shared: versions fail on the same questions
    order = send_order(QUESTIONS, grouped=True)
    out = []

    def live(name, q, cache):
        model, layout, rate, ttft, speed, length = VERSIONS[name]
        hard = not q["prepared"]["judge"]["pass"]
        p = 1 / (1 + math.exp(-(logit(rate) + lesson_ease[q["lesson_id"]] - 1.6 * hard)))
        size = total_of[(q["lesson_id"], q["key"])]
        output = max(40, int(t.estimate_tokens(q["prepared"]["a"]) * length * rng.lognormvariate(0, 0.2)))
        read, write = (cache["cache_read"], cache["cache_write"]) if layout == "cacheable" else (0, 0)
        first = ttft * (1 - 0.3 * read / size) * rng.lognormvariate(0, 0.45) * (3 if rng.random() < 0.05 else 1)  # 1 in 20 waits behind a busy gateway
        return {"passed": luck[(q["lesson_id"], q["key"])] < p, "input_tokens": size - read - write, "cache_read": read, "cache_write": write,
                "output_tokens": output, "first_words_s": first, "total_s": first + output * 0.75 / speed,
                "cost": answer_cost(model, size - read - write, output, read, write), "model": model}

    for name, (model, layout, *_rest) in VERSIONS.items():
        caching = t.cache_pass(order, start_of if layout == "cacheable" else dict.fromkeys(start_of, 0), minimum=t.min_cacheable(model))
        for q, cache in zip(QUESTIONS, caching):
            if name.startswith("prepared"):
                if q["prepared"]["judge"]["pass"]:   # shown at once, already paid for
                    row = {"passed": True, "input_tokens": 0, "cache_read": 0, "cache_write": 0, "output_tokens": 0,
                           "first_words_s": rng.uniform(0.03, 0.12), "total_s": 0.12, "cost": 0.0, "model": "prepared"}
                else:                                # a failed prepared answer is never shown: the live tutor answers instead
                    row = live("Sonnet", q, cache)
            else:
                row = live(name, q, cache)
            out.append({"version": name, "lesson_id": q["lesson_id"], "key": q["key"], **row})
    return pd.DataFrame(out)


REC = simulate()
REC["source"] = "practice"
print(f"{len(REC):,} answers: {REC.version.nunique()} versions x {len(QUESTIONS)} questions")
REC.head()

**What that cell did.** It produced one row per version per question, with the columns a real run
records: passed, input, cached and output tokens, seconds to first words, total seconds, cost. The
**prepared** version shows its answer if the judge passed it. If the judge failed it, the page never
shows it and asks the live tutor instead, here Sonnet, which pays for that answer. That's exactly what
the app does.

## 4. Speed: the slow case, not the average

> **In plain words.** Averages hide the slow answers, and those are the ones learners notice. So compare
> **p95**, the time 95 out of 100 answers arrive within (lesson "Latency that users feel"). For a chat,
> the time to the **first words** matters most, because once text starts appearing, people read along.

**Exercise.** Write `percentile`: sort the times, then take the one at position `ceil(n * p / 100)`,
counting from 1 (so subtract 1 for Python's index). Never go below position 1. An empty list gives 0.

In [ ]:
def percentile(times, p):
    """The time that p out of 100 requests finish within (nearest rank)."""
    # TODO
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_median_of_ten():
    assert percentile([5, 1, 9, 3, 7, 2, 8, 4, 10, 6], 50) == 5


def test_p95_of_ten_is_the_slowest():
    assert percentile(list(range(1, 11)), 95) == 10


def test_one_time():
    assert percentile([3.0], 95) == 3.0


def test_no_times():
    assert percentile([], 95) == 0.0

In [ ]:
speed = REC.groupby("version", sort=False).agg(
    first_words_p50=("first_words_s", lambda s: percentile(list(s), 50)),
    first_words_p95=("first_words_s", lambda s: percentile(list(s), 95)),
    total_p95=("total_s", lambda s: percentile(list(s), 95)))
speed.round(2)

**What to look for.** The gap between the typical case (p50) and the slow one (p95). One answer in 20
here waits behind a busy gateway, which is what pulls the p95 up. Prepared answers are near zero most of
the time, but their p95 isn't, because the failed ones fall back to a live answer.

## 5. Quality: no worse than today, on the same questions

> **In plain words.** Every version answered the **same** questions, so compare them question by
> question. Only the questions where two versions differ tell you anything (lesson "Telling real gains
> from noise"). The question isn't "is it better?" but "is it **no worse** than today, by more than 3
> points?" That's a **non-inferiority** test. A cheaper version only needs to show it doesn't lose much.
>
> `t.paired_difference` gives the new pass rate minus the baseline's, with a 95% interval from
> resampling **whole lessons**. Answers about the same lesson aren't independent, so resampling single
> answers would make the interval look narrower than it really is. The rule passes a version when the
> **lower end** of the interval is above -3 points.

In [ ]:
verdicts = {v: {(r.lesson_id, r.key): bool(r.passed) for r in grp.itertuples()} for v, grp in REC.groupby("version", sort=False)}
quality = []
for v in VERSIONS:
    _, lo, hi = tpe.wilson(sum(verdicts[v].values()), len(verdicts[v]))
    diff, dlo, dhi = t.paired_difference(verdicts[BASELINE], verdicts[v])
    quality.append({"version": v, "pass_rate": sum(verdicts[v].values()) / len(verdicts[v]), "wilson_low": lo, "wilson_high": hi,
                    "vs_baseline": diff, "diff_low": dlo, "diff_high": dhi})
quality = pd.DataFrame(quality).set_index("version")
quality.round(3)

**What to look for.** The `diff_low` column. A version whose interval reaches well below -0.03 might be
worse than today by more than the margin, so it's out, however cheap it is. The baseline compared with
itself is exactly 0. Note the Wilson interval treats every answer as independent. It's printed only so
you can see it's narrower than the lesson-level one.

**Careful with the prepared version.** Its verdicts are real, but they come from the same judge that
decided which prepared answers to keep. So it's graded by the judge that chose it, and that flatters it.
In the real run, every version's answers, prepared ones included, are judged again by one judge. Make
that judge a different model from the one that did the choosing.

## 6. All three scores together, and the versions nothing beats

> **In plain words.** Put quality, cost and speed side by side. Then throw out every version that another
> version **beats**: at least as good on all three, and better on at least one. What's left is the
> **Pareto front**, the real trade-offs (lesson "Comparing LLMs for a task").

`t.beats(a, b)` says whether `a` beats `b`, using the `quality`, `cost` and `p95` of each.

**Exercise.** Write `pareto`: keep the name of each version that **no other** version beats. Compare
each version with every other one, but never with itself.

In [ ]:
def pareto(versions):
    """The names of the versions nothing beats."""
    # TODO: use t.beats(other, v)
    raise NotImplementedError

In [ ]:
%%ipytest -qq

VS = [{"name": "a", "quality": .9, "cost": 2, "p95": 3}, {"name": "b", "quality": .8, "cost": 1, "p95": 3},
      {"name": "c", "quality": .8, "cost": 3, "p95": 4}, {"name": "d", "quality": .9, "cost": 2, "p95": 3}]


def test_beaten_on_everything_is_dropped():
    assert "c" not in pareto(VS)


def test_each_wins_on_something():
    assert {"a", "b"} <= set(pareto(VS))


def test_a_tie_beats_neither():
    assert "a" in pareto(VS) and "d" in pareto(VS)

In [ ]:
table = pd.DataFrame({
    "quality": quality.pass_rate, "diff_low": quality.diff_low,
    "cost_per_1000": REC.groupby("version", sort=False).cost.mean() * 1000,
    "p95": speed.first_words_p95,
    "cache_hit_rate": pd.Series({v: hit_rate(g.to_dict("records")) for v, g in REC.groupby("version", sort=False)}),
})
versions = [{"name": v, "quality": r.quality, "cost": r.cost_per_1000, "p95": r.p95, "diff_low": r.diff_low} for v, r in table.iterrows()]
front = pareto(versions)
table["on_front"] = [v in front for v in table.index]
table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for v, r in table.iterrows():
    slow = r.p95 > P95_LIMIT
    ax.scatter(r.cost_per_1000, r.quality, s=80, marker="x" if slow else "o", alpha=1 if r.on_front else .35)
    below = "cacheable" in v  # its twin sits right next to it: label one above, one below
    ax.annotate(v + (" (too slow)" if slow else ""), (r.cost_per_1000, r.quality), textcoords="offset points",
                xytext=(6, -12 if below else 5), fontsize=8)
ax.axhline(table.loc[BASELINE, "quality"] - MARGIN, linestyle=":", linewidth=1)
ax.text(ax.get_xlim()[1], table.loc[BASELINE, "quality"] - MARGIN, "baseline - margin ", va="bottom", ha="right", fontsize=8)
ax.set_xlabel("dollars per 1,000 answers (lower is better)")
ax.set_ylabel("pass rate (higher is better)")
ax.set_title("Practice recording: faded = beaten by another version")
plt.show()

**What to look for.** Up and to the left is better. The faded points are beaten by something, so they
are never the right choice.

Look at the cacheable versions next to the ones they came from. On the Claude 5.5 models every lesson's
start is long enough, so most of their input comes from the cache. They're about a third cheaper, and a
little quicker to the first words, so they **beat** the originals outright. The saving isn't bigger
because output is most of the bill, and the first call on each lesson pays a quarter extra to write the
cache. The practice recording gives them the same quality as the originals, which is an assumption. In
the real run, check it: the model now reads the rules before the question.

The big move is **prepared, else Sonnet**. Most standard questions cost nothing to answer and come back
at once. Only the few prepared answers that failed their check pay for a live one.

## 7. Let the rule decide

**Exercise.** Write `choose`, the rule from section 0, in order:

1. drop every version whose `diff_low` is below `-margin` (reason `"quality"`);
2. of the rest, drop every version whose `p95` is above `p95_limit` (reason `"speed"`);
3. the choice is the cheapest version left (if two cost the same, the faster one), or `None` if none is left.

Return `{"choice": name or None, "kept": [names left, in order], "dropped": {name: reason}}`.

In [ ]:
def choose(versions, margin, p95_limit):
    """Quality first, then speed, then the cheapest of what's left."""
    # TODO
    raise NotImplementedError

In [ ]:
%%ipytest -qq

CANDIDATES = [{"name": "base", "diff_low": 0.0, "p95": 4, "cost": 5},
              {"name": "cheap-but-worse", "diff_low": -0.08, "p95": 2, "cost": 1},
              {"name": "cheap-and-slow", "diff_low": -0.01, "p95": 9, "cost": 1},
              {"name": "cheaper", "diff_low": -0.02, "p95": 3, "cost": 2}]


def test_the_cheapest_that_holds_quality_and_speed():
    assert choose(CANDIDATES, margin=0.03, p95_limit=5)["choice"] == "cheaper"


def test_it_says_why_each_was_dropped():
    assert choose(CANDIDATES, margin=0.03, p95_limit=5)["dropped"] == {"cheap-but-worse": "quality", "cheap-and-slow": "speed"}


def test_nothing_left():
    assert choose(CANDIDATES[1:3], margin=0.03, p95_limit=5)["choice"] is None

In [ ]:
decision = choose(versions, MARGIN, P95_LIMIT)
print("kept:   ", decision["kept"])
print("dropped:", decision["dropped"])
print("choice: ", decision["choice"], " <- from the practice recording: the method, not the answer")

**What to look for.** The rule says *why* each version lost, which is the part to show a manager. Note
what it doesn't do: it never looks at the chart and picks a favourite. If you disagree with the choice,
change the plan in section 0 and say so. Don't change the result.

To see what the money is worth, the course's lesson "Net value per month" adds the other side: the value
of a good answer. A version that costs more can still be the better buy if it helps more learners.

## 8. Before the real run: how many questions, and what it costs

> **In plain words.** A real run sends every question to every version, and then a judge grades every
> answer. Both cost money. So first work out how many questions you need to tell a 3-point drop from
> luck (lesson "How many examples do you need?"), then what that many will cost.

For a paired non-inferiority test, a common rule of thumb is: questions ≈ (1.645 + 0.84)² ×
(share of questions where the two versions differ) ÷ margin². That's 95% sure, with an 80% chance of
spotting a real drop. Answers come in lessons, so add about half again: lesson-level intervals are wider,
as the judging notebook's section 6 shows.

The rule assumes the two versions are really equal, which is the hardest case to call. A version that is
clearly better or worse shows it with far fewer questions. So size the run for the **close calls**,
the versions within a few points of the baseline.

In [ ]:
differ = {v: sum(verdicts[v][k] != verdicts[BASELINE][k] for k in verdicts[BASELINE]) / len(verdicts[BASELINE]) for v in VERSIONS if v != BASELINE}
needed = {v: math.ceil((1.645 + 0.84) ** 2 * d / MARGIN ** 2 * 1.5) for v, d in differ.items()}
close = [v for v in needed if abs(quality.loc[v, "vs_baseline"]) <= 2 * MARGIN and needed[v]]
n = min(max([needed[v] for v in close] or [len(QUESTIONS)]), len(QUESTIONS))
print("questions needed per version:", needed)
print("close calls:", close, "(0 needed means the practice recording made two versions answer alike)")
print(f"-> run {n} questions (the dev set has {len(QUESTIONS)})")

# Judging, from the real drafting run: Sonnet judged 2,094 answers for 3,429,008 input and 2,212,630 output tokens
judge_each = tp.cost(SONNET, 3_429_008, 2_212_630) / 2094
answers_each = REC.groupby("version", sort=False).cost.mean()
run = pd.DataFrame({"answers": answers_each * n, "judging": judge_each * n})
run.loc["total"] = run.sum()
run.round(2)

**What to look for.** The total in the last row, and which part drives it. Judging every answer with a
strong model often costs more than the answers themselves. Two ways to spend less: judge with a cheaper
model, once you've checked you can trust it (the judging notebook), or drop versions the plan already
rules out. The real runner prints this estimate and asks before it spends anything.

## 9. Track it in MLflow

> **In plain words.** One **run** per version, all inside one parent run for the experiment. Each run
> logs its settings (params), its scores (metrics) and a `source` tag, so a practice run can never be
> mistaken for a real one. The parent run holds the plan, the decision, the chart and every table behind
> them. Every run gets a name you can read at a glance and a description (the run's **note** in MLflow)
> saying what question it answers, what ran, how it's measured and how to reproduce it.
>
> By default this uses a **throwaway** store in a temporary folder, so it costs nothing and leaves no
> trace. Set `USE_LOCAL_SERVER = True` to log to your MLflow at `http://localhost:5050` instead, in the
> `daily-drill` experiment, and keep the runs.

In [ ]:
import os
import tempfile

os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")
os.environ.setdefault("MLFLOW_GENAI_EVAL_MAX_WORKERS", "4")   # a local SQLite store can't take many writers at once
import mlflow

USE_LOCAL_SERVER = False   # True: keep these runs in your MLflow (http://localhost:5050, experiment daily-drill)
if USE_LOCAL_SERVER:
    mlflow.set_tracking_uri("http://localhost:5050")
    mlflow.set_experiment("daily-drill")
else:
    mlflow.set_tracking_uri(f"sqlite:///{tempfile.mkdtemp(prefix='tradeoff-mlflow-')}/mlflow.db")
    mlflow.set_experiment("tutor-cost-speed-quality-learning")
NOTEBOOK = "notebooks/learning/cost_speed_quality.ipynb"


def note(question, what_ran, measured, reproduce):
    """A run's description, the same four parts on every run."""
    return (f"**Question:** {question}\n\n**What ran:** {what_ran}\n\n**How it's measured:** {measured}\n\n"
            f"**Reproduce:** {reproduce}")


with mlflow.start_run(run_name=f"tutor versions · cost, speed and quality · practice recording · dev-{len(QUESTIONS)}") as parent:
    mlflow.set_tags({"split": "dev", "source": "practice", "purpose": "learning", "decision": str(decision["choice"]),
                     "question": "which version of the tutor answers the standard questions",
                     "mlflow.note.content": note(
                         "which version of the tutor should answer the standard questions?",
                         f"{len(VERSIONS)} versions on the same {len(QUESTIONS)} dev questions: a practice recording built on the "
                         "real prompts and prepared answers, with made-up live models (see the notebook's section 3). "
                         "Real Foundry runs are nested here too (section 10).",
                         f"pass rate with a lesson-level paired interval against {BASELINE}, p95 time to first words, dollars per "
                         f"1,000 answers with the prompt cache counted. Rule: {PLAN['decision_rule']}",
                         f"run {NOTEBOOK} with USE_LOCAL_SERVER = True."),
                     })
    mlflow.log_params({"baseline": BASELINE, "margin": MARGIN, "p95_limit_s": P95_LIMIT, "questions": len(QUESTIONS),
                       "min_cacheable": t.MIN_CACHEABLE, "decision_rule": PLAN["decision_rule"]})
    mlflow.log_dict(PLAN, "plan.json")
    mlflow.log_dict(decision, "decision.json")
    mlflow.log_figure(fig, "front.png")
    # the analysis behind the decision, not just the chart: every table a reviewer would ask for
    mlflow.log_table(quality.reset_index().round(4).to_dict(orient="list"), "quality_paired_intervals.json")
    mlflow.log_table(speed.reset_index().round(3).to_dict(orient="list"), "speed_percentiles.json")
    mlflow.log_table(table.reset_index().round(4).to_dict(orient="list"), "versions_side_by_side.json")
    mlflow.log_table(cache_table.reset_index().rename(columns=str).round(3).to_dict(orient="list"), "cache_hit_rate_by_order.json")
    mlflow.log_table(starts.to_dict(orient="list"), "fixed_start_tokens.json")
    mlflow.log_table(run.reset_index().rename(columns={"index": "version"}).round(2).to_dict(orient="list"), "real_run_estimate.json")
    mlflow.log_dict({"questions_needed": needed, "close_calls": close, "questions_to_run": n}, "sample_size.json")
    for v, r in table.iterrows():
        model, layout, *_ = VERSIONS[v]
        with mlflow.start_run(run_name=f"version {v} · practice · dev-{len(QUESTIONS)}", nested=True):
            mlflow.set_tags({"split": "dev", "source": "practice", "on_front": str(r.on_front),
                             "mlflow.note.content": note(
                                 f"how does the version '{v}' score on quality, speed and cost?",
                                 f"{model} with {layout} prompt layout, from the practice recording (made-up quality and timing on "
                                 "real prompts; see section 3).",
                                 "pass rate and its paired difference with the baseline, p95 time to first words, cost per 1,000 "
                                 "answers, cache hit rate.",
                                 f"run {NOTEBOOK}, section 9.")})
            mlflow.log_params({"model": model, "prompt": layout})
            mlflow.log_metrics({"pass_rate": r.quality, "diff_low": r.diff_low, "cost_per_1000_usd": r.cost_per_1000,
                                "first_words_p95_s": r.p95, "cache_hit_rate": r.cache_hit_rate})
found = mlflow.search_runs(filter_string=f"tags.mlflow.parentRunId = '{parent.info.run_id}'")
found[["tags.mlflow.runName", "params.model", "params.prompt", "metrics.pass_rate", "metrics.cost_per_1000_usd", "metrics.first_words_p95_s"]].round(3)

## 10. On Microsoft Foundry: a few calls inline, then the rest as a batch

> **In plain words.** There are two ways to send work to a model. **Inline** is one call at a time while
> you wait: it's what a learner gets, so it's the only way to measure speed. A **batch** is many
> requests in one file, sent as one job that comes back later (the target is 24 hours, often it's minutes) at about
> **half the price**. Nobody waits on an eval set, so a batch suits answering and judging it, but it
> can't tell you how fast the model answers a person.

So a real run uses both: a **small inline sample** for speed, and a **batch** for quality and cost on
every question. On Foundry, batch needs a deployment of type `GlobalBatch`. Only OpenAI models offer
it, so this section uses GPT-5.4 mini to answer and GPT-5.4 to judge. GPT-5.4 is a different model
family from the one that chose the prepared answers, which keeps the judge fair (section 5).

**Where it fits in the MLflow experiment.** A batch outlives the notebook session that started it, so
the **run** is what remembers it:

| Moment | What MLflow keeps |
|---|---|
| Inline sample | a run with one **trace** per call, scored for time to first words and cost |
| Batch sent | a run with the **batch id** and a `stage` tag, plus the request file as an artifact |
| Following up (later, maybe another day) | the same run, **resumed by its id**: the stage moves on, the output file is logged |
| Answers back | an evaluation on the **finished outputs**: one trace per answer, nothing called again |
| Verdicts back | each verdict attached to its trace as **feedback**, and the pass rate logged as a metric |

`RUN_LIVE = False` replays a real run recorded on 2026-10-18 (`foundry_batch_demo.json`), so it costs
nothing. To run it yourself: `az login`, set `AZURE_OPENAI_ENDPOINT` to your Foundry resource, and set
the three deployment names below (they never go in the repo).

In [ ]:
import json
import os
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

from drill import foundry_batch as fb
from mlflow.entities import AssessmentSource, Feedback
from mlflow.genai.scorers import scorer

RUN_LIVE = False   # True: real calls on your Foundry resource. Costs money (a few cents for this sample)
DEMO_FILE = Path(tp.__file__).parent / "content" / "prepared_eval" / "foundry_batch_demo.json"
FB_REC = json.loads(DEMO_FILE.read_text())
by_cid = {fb.cid(q): q for q in QUESTIONS}
DEMO = [by_cid[c] for c in FB_REC["questions"]]     # 20 dev questions, picked at random (seed 7)
INLINE_QS = DEMO[:5]                                # the inline sample: the first 5 of them

if RUN_LIVE:
    api = fb.client()                               # signed in as you: no key
    INLINE_DEP = os.environ["DRILL_INLINE_DEPLOYMENT"]       # a GlobalStandard deployment of gpt-5.4-mini
    ANSWER_DEP = os.environ["DRILL_BATCH_ANSWER_DEPLOYMENT"]  # a GlobalBatch deployment of gpt-5.4-mini
    JUDGE_DEP = os.environ["DRILL_BATCH_JUDGE_DEPLOYMENT"]    # a GlobalBatch deployment of gpt-5.4
print(f"{'LIVE' if RUN_LIVE else 'replaying the run recorded on ' + FB_REC['recorded']}: "
      f"{len(INLINE_QS)} inline, {len(DEMO)} in the batch")

### 10a. The inline sample: timed, one call at a time

Each call goes through `mlflow.genai.evaluate`, so each one becomes a **trace** in a run of its own,
nested under the experiment's parent run from section 9. Two code scorers record what a person would
feel and pay: seconds to the first words, and dollars. Quality comes later, from the judge batch.

In [ ]:
inline = {}


def answer_inline(lesson_id, key):
    """The system under test, inline: one streamed call (or its recording), timed."""
    cid = f"{lesson_id}/{key}"
    if RUN_LIVE:
        r = fb.ask_inline(api, INLINE_DEP, t.today_messages(DATA, lesson_id, by_cid[cid]["chip"]))
        r["cost"] = fb.cost("gpt-5.4-mini", "standard", r["usage"])
    else:
        r = FB_REC["inline"][cid]
    inline[cid] = r
    return r["text"]


@scorer(name="first_words_s")
def first_words_s(inputs):
    return inline[f"{inputs['lesson_id']}/{inputs['key']}"]["first_words_s"]


@scorer(name="cost_usd")
def cost_usd(inputs):
    return inline[f"{inputs['lesson_id']}/{inputs['key']}"]["cost"]


with mlflow.start_run(run_id=parent.info.run_id):                     # back inside the experiment's parent run
    with mlflow.start_run(run_name=f"answer gpt-5.4-mini · foundry inline · tutor Ask · dev-{len(INLINE_QS)}", nested=True) as inline_run:
        mlflow.set_tags({"split": "dev", "source": "live" if RUN_LIVE else "recording", "mode": "inline",
                         "mlflow.note.content": note(
                             "how fast, and at what cost, does GPT-5.4 mini answer the tutor's questions when a learner waits?",
                             f"{len(INLINE_QS)} dev questions, the tutor's own prompt, streamed one at a time on Microsoft Foundry"
                             + ("" if RUN_LIVE else f" (a replay of the run recorded {FB_REC['recorded']})") + ".",
                             "seconds to the first words and in all, tokens and dollars per answer; quality comes later from "
                             "the judge batch, as feedback on each trace.",
                             f"run {NOTEBOOK}, section 10, with RUN_LIVE = True.")})
        mlflow.log_params({"model": "gpt-5.4-mini", "mode": "inline", "questions": len(INLINE_QS)})
        mlflow.genai.evaluate(data=[{"inputs": {"lesson_id": q["lesson_id"], "key": q["key"]}} for q in INLINE_QS],
                              predict_fn=answer_inline, scorers=[first_words_s, cost_usd])
        times = [r["first_words_s"] for r in inline.values()]
        mlflow.log_metrics({"first_words_p50_s": percentile(times, 50), "first_words_p95_s": percentile(times, 95),
                            "cost_per_1000_usd": 1000 * sum(r["cost"] for r in inline.values()) / len(inline)})

for cid, r in inline.items():
    print(f"{cid:40} first words {r['first_words_s']:5.2f}s, all {r['total_s']:5.2f}s, "
          f"{r['usage'].get('completion_tokens', 0):4} tokens out, ${r['cost']:.5f}")

**What to look for.** Real times, from a real gateway, for the model a learner would wait on. With only
five calls the p95 is just the slowest one, so a real run times 100 or more, spread over the day.

### 10b. The batch: one file, one job

A batch file has one request a line. Each line carries a `custom_id`, the address its answer comes back
to, plus the same body an inline call would send. Look at the first line. Then the job is sent, and the
run records its id and stage. The run **ends** here on purpose: the job carries on without us.

In [ ]:
rows = fb.lines(fb.answer_requests(DATA, DEMO), "<batch deployment>" if not RUN_LIVE else ANSWER_DEP)
first = rows[0]
print(json.dumps({**first, "body": {**first["body"], "messages": [{"role": "user", "content": first["body"]["messages"][0]["content"][:160] + " ..."}]}}, indent=1))

with mlflow.start_run(run_id=parent.info.run_id):
    with mlflow.start_run(run_name=f"answer gpt-5.4-mini · foundry batch · tutor Ask · dev-{len(DEMO)}", nested=True) as batch_run:
        mlflow.set_tag("mlflow.note.content", note(
            "what do the same answers cost as one batch job, and how good are they?",
            f"{len(DEMO)} dev questions as one batch on Microsoft Foundry, followed up from this run in later sessions (section "
            "10c), then judged by GPT-5.4 in a second batch with the judge fitted to live answers"
            + ("" if RUN_LIVE else f" (a replay of the run recorded {FB_REC['recorded']})") + ".",
            "pass rate from the judge's verdicts (feedback on each trace), dollars per 1,000 answers; batch has no speed to "
            "measure, since answers wait in a queue.",
            f"run {NOTEBOOK}, section 10, with RUN_LIVE = True."))
        batch_id = fb.submit(api, rows, note="tutor tradeoff, dev") if RUN_LIVE else "batch_replay_answer"
        sent = datetime.now(timezone.utc)
        mlflow.set_tags({"split": "dev", "source": "live" if RUN_LIVE else "recording", "mode": "batch",
                         "stage": "answers_sent", "answer_batch_id": batch_id, "resends": "0",
                         "submitted_at": sent.isoformat(timespec="seconds"),
                         "due_at": (sent + timedelta(hours=24)).isoformat(timespec="seconds"),
                         "git_commit": tpe.git_commit()})
        mlflow.log_params({"model": "gpt-5.4-mini", "judge": "gpt-5.4", "mode": "batch", "questions": len(DEMO)})
        mlflow.log_text(fb.jsonl(rows).decode(), "requests.jsonl")
        # what the requests were built from, so a late result is read against the right lesson and price
        mlflow.log_dict({fb.cid(q): tp.fingerprint(DATA, q["lesson_id"], q["chip"]) for q in DEMO}, "fingerprints.json")
        mlflow.log_dict({f"{m} {mode}": p for (m, mode), p in fb.PRICES.items()}, "prices.json")
BATCH_RUN_ID = batch_run.info.run_id
print(f"\nsent {len(rows)} requests as {batch_id}; MLflow run {BATCH_RUN_ID} remembers it")

### 10c. Following up: run this again until it says `judged`

This is the part people get wrong. Nobody sits and waits for a batch, so the follow-up must work from
**what MLflow remembers**, not from variables in memory. Live, you'd run the cell below after lunch, or
tomorrow in a fresh session: it only needs `BATCH_RUN_ID`. Each time it resumes the run, checks the
job, and moves it one stage on:

`answers_sent` → (answers back) `judge_sent` → (verdicts back) `judged`

**The 24-hour window.** A batch aims to finish within 24 hours, not now, so plan for it. Answers and
then judging is two batches in a row: allow two days before the decision (it's in the plan, section 0).
Three things in the cell below come from that:

- **The clock.** The run records when each job was sent and when it's due (24 hours later). Each check
  says how much time is left, or how late the job is.
- **Late jobs.** On Foundry, 24 hours is a *target*: a late job isn't stopped, it keeps running until it
  finishes or you **cancel** it. Cancelling returns what already finished, and you pay only for that.
  (OpenAI's own API is different: there, a job still running after its window ends as `expired`, with
  the same partial results.) Either way, the follow-up sends the leftovers as a new job, records the new
  id on the same run, and waits again.
- **Things that change while you wait.** The run keeps each lesson's fingerprint and the prices used.
  An answer whose lesson was edited before the results came back is stale, so it's dropped, not scored.

When answers come back, it reads the output file and matches each line to its request by `custom_id`
(the lines can come back in any order, and some can fail). Then it logs an evaluation of the finished
answers, with no model called again, and sends the judge batch. When the verdicts come back, it
attaches each one to its trace as **feedback**.

**Exercise.** Write `match_by_id`: given the request `rows` and the output lines (each a dict with a
`custom_id`, and either `text` or `error`), return `{custom_id: text}` for **every** request. Use `None`
for a request that failed or never came back. Ignore output lines whose `custom_id` isn't one of the
requests. Keep the requests' order.

In [ ]:
def match_by_id(rows, outputs):
    """Each request's answer by its custom_id, or None if it failed or never came back."""
    # TODO: index the outputs by custom_id, then walk the rows in order
    raise NotImplementedError

In [ ]:
%%ipytest -qq

ROWS = [{"custom_id": "a"}, {"custom_id": "b"}, {"custom_id": "c"}]


def test_out_of_order_lines_find_their_request():
    outs = [{"custom_id": "c", "text": "C"}, {"custom_id": "a", "text": "A"}, {"custom_id": "b", "text": "B"}]
    assert match_by_id(ROWS, outs) == {"a": "A", "b": "B", "c": "C"}


def test_failed_and_missing_are_none():
    outs = [{"custom_id": "a", "text": "A"}, {"custom_id": "b", "error": "rate limited"}]
    assert match_by_id(ROWS, outs) == {"a": "A", "b": None, "c": None}


def test_strangers_are_ignored_and_order_kept():
    outs = [{"custom_id": "zzz", "text": "?"}, {"custom_id": "b", "text": "B"}]
    assert list(match_by_id(ROWS, outs)) == ["a", "b", "c"]

In [ ]:
def job_status(step, batch_id, run_tags):
    """One check on a job: live from Foundry, or the next recorded check (counted in a run tag)."""
    if RUN_LIVE:
        check = fb.snapshot(api.batches.retrieve(batch_id))
        seen = json.loads(run_tags.get(f"{step}_checks", "[]"))
        if not seen or seen[-1]["status"] != check["status"]:        # keep each change of status, for the recording
            started = datetime.fromisoformat(run_tags.get(f"{step}_due_at", run_tags["due_at"])) - timedelta(hours=24)
            seen.append({**check, "after_s": round((datetime.now(timezone.utc) - started).total_seconds())})
            mlflow.set_tag(f"{step}_checks", json.dumps(seen))
        return check
    checks = FB_REC["jobs"][step]["checks"]
    n = int(run_tags.get(f"{step}_checks_seen", 0))
    mlflow.set_tag(f"{step}_checks_seen", n + 1)
    return checks[min(n, len(checks) - 1)]


def job_output(step, batch_id):
    """The finished job's output lines, as {custom_id, text} or {custom_id, error}."""
    if RUN_LIVE:
        return [{"custom_id": k, **v} for k, v in fb.read_output(api, api.batches.retrieve(batch_id)).items()]
    if step == "answer":
        return [{"custom_id": k, **v} for k, v in FB_REC["batch"].items()]
    return [{"custom_id": k, "text": json.dumps(v)} for k, v in FB_REC["verdicts"].items()]


def follow_up(run_id):
    """Resume the batch's run, check its job, and move it one stage on if the job is done."""
    tags = mlflow.get_run(run_id).data.tags
    stage = tags["stage"]
    if stage == "judged":
        return stage
    step, bid = ("answer", tags["answer_batch_id"]) if stage == "answers_sent" else ("judge", tags["judge_batch_id"])
    with mlflow.start_run(run_id=run_id):
        check = job_status(step, bid, tags)
        left = datetime.fromisoformat(tags[f"{step}_due_at" if f"{step}_due_at" in tags else "due_at"]) - datetime.now(timezone.utc)
        hours = left.total_seconds() / 3600
        clock = "" if not RUN_LIVE else f", {hours:.0f} of 24 hours left" if hours >= 0 else (
            f", {-hours:.0f} hours past the 24-hour target: to stop waiting, api.batches.cancel('{bid}') keeps what finished")
        print(f"{step} job: {check['status']}, {check['completed']}/{check['total']} done, {check['failed']} failed{clock}")
        if check["status"] not in fb.DONE:
            return stage                                            # not yet: come back later
        out = job_output(step, bid)
        sent_rows = fb.lines(fb.answer_requests(DATA, DEMO), "-") if step == "answer" else None
        missing = [r for r in (sent_rows or []) if not match_by_id([r], out)[r["custom_id"]]]
        if missing and RUN_LIVE and int(tags.get("resends", 0)) < 2:  # cancelled, expired or failed: send what's left as a new job
            for r in missing:
                r["body"]["model"] = ANSWER_DEP
            mlflow.log_text("".join(json.dumps(o) + "\n" for o in out), f"{step}_output_part{tags.get('resends', 0)}.jsonl")
            new_id = fb.submit(api, missing, note="tutor tradeoff, leftovers")
            mlflow.set_tags({"answer_batch_id": new_id, "resends": str(int(tags.get("resends", 0)) + 1),
                             "answer_due_at": (datetime.now(timezone.utc) + timedelta(hours=24)).isoformat(timespec="seconds")})
            print(f"{len(missing)} not answered ({check['status']}): sent again as {new_id}. Follow up later.")
            return stage
        mlflow.log_text("".join(json.dumps(o) + "\n" for o in out), f"{step}_output.jsonl")
        if step == "answer":
            texts = match_by_id(sent_rows, out)
            then = mlflow.artifacts.load_dict(f"runs:/{run_id}/fingerprints.json")
            changed = [c for c in texts if then.get(c) != tp.fingerprint(DATA, by_cid[c]["lesson_id"], by_cid[c]["chip"])]
            for c in changed:                                       # the lesson changed while we waited: that answer is stale
                texts[c] = None
            if changed:
                print(f"dropped {len(changed)} answers whose lesson changed since the batch was sent")
            usage = {o["custom_id"]: o.get("usage", {}) for o in out}

            @scorer(name="cost_usd")
            def batch_cost(inputs):
                return fb.cost("gpt-5.4-mini", "batch", usage.get(f"{inputs['lesson_id']}/{inputs['key']}", {}))

            mlflow.genai.evaluate(                                  # finished outputs: no predict_fn, nothing re-run
                data=[{"inputs": {"lesson_id": q["lesson_id"], "key": q["key"]}, "outputs": texts[fb.cid(q)] or ""} for q in DEMO],
                scorers=[batch_cost])
            # one judge job for the batch's answers and the inline sample's, so both get the same judge
            judge_rows = fb.lines(fb.judge_requests(DATA, DEMO, texts), "-", max_tokens=2000) + fb.lines(
                [("inline:" + k, m) for k, m in fb.judge_requests(DATA, INLINE_QS, {k: r["text"] for k, r in inline.items()})], "-", max_tokens=2000)
            if RUN_LIVE:   # also the prepared answers' strict checklist, in the same job, to compare the two judges
                inline_texts = {k: r["text"] for k, r in inline.items()}
                judge_rows += fb.lines([("strict:" + k, m) for k, m in fb.judge_requests(DATA, DEMO, texts, live=False)]
                                       + [("strict:inline:" + k, m) for k, m in fb.judge_requests(DATA, INLINE_QS, inline_texts, live=False)],
                                       "-", max_tokens=2000)
                for r in judge_rows:
                    r["body"]["model"] = JUDGE_DEP
            judge_id = fb.submit(api, judge_rows, note="tutor tradeoff judge") if RUN_LIVE else "batch_replay_judge"
            mlflow.log_metrics({"answered": len([v for v in texts.values() if v]), "failed": len([v for v in texts.values() if not v]),
                                "cost_per_1000_usd": 1000 * sum(fb.cost("gpt-5.4-mini", "batch", u) for u in usage.values()) / len(DEMO)})
            mlflow.set_tags({"stage": "judge_sent", "judge_batch_id": judge_id,
                             "judge_due_at": (datetime.now(timezone.utc) + timedelta(hours=24)).isoformat(timespec="seconds")})
            return "judge_sent"
        verdicts = {cid: fb.verdict(text) if RUN_LIVE else json.loads(text) for cid, text in match_by_id(
            [{"custom_id": o["custom_id"]} for o in out], out).items() if text}
        strict = {k.removeprefix("strict:"): v for k, v in verdicts.items() if k.startswith("strict:")}
        verdicts = {k: v for k, v in verdicts.items() if not k.startswith("strict:")}
        attach_verdicts(run_id, {k: v for k, v in verdicts.items() if not k.startswith("inline:")})
        attach_verdicts(inline_run.info.run_id, {k.removeprefix("inline:"): v for k, v in verdicts.items() if k.startswith("inline:")})
        mlflow.set_tag("stage", "judged")
        if RUN_LIVE:   # the recording the notebook replays, written by this run from what it logged
            write_recording(run_id, verdicts, strict, out)
        return "judged"


def write_recording(run_id, verdicts, strict, judge_out):
    tags = mlflow.get_run(run_id).data.tags
    answers_out = [json.loads(line) for line in mlflow.artifacts.load_text(f"runs:/{run_id}/answer_output.jsonl").splitlines()]
    usage_by = {o["custom_id"]: o.get("usage", {}) for o in judge_out}
    rec = {"about": f"Section 10 of {NOTEBOOK}, run live: GPT-5.4 mini on Microsoft Foundry, {len(INLINE_QS)} inline calls, then "
                    f"{len(DEMO)} questions as one batch, then one GPT-5.4 judge batch (fitted to live answers). Written by MLflow runs "
                    f"{inline_run.info.run_id} and {run_id}. Endpoint and resource names are left out on purpose.",
           "recorded": datetime.now(timezone.utc).strftime("%Y-%m-%d"), "inline_run_id": inline_run.info.run_id, "batch_run_id": run_id,
           "inline_model": "gpt-5.4-mini", "batch_model": "gpt-5.4-mini", "judge_model": "gpt-5.4",
           "questions": [fb.cid(q) for q in DEMO], "inline": inline,
           "batch": {o["custom_id"]: {k: v for k, v in o.items() if k != "custom_id"} | {"cost": fb.cost("gpt-5.4-mini", "batch", o.get("usage", {}))}
                     for o in answers_out},
           "verdicts": {c: {**v, "cost": fb.cost("gpt-5.4", "batch", usage_by.get(c, {}))} for c, v in verdicts.items()},
           "verdicts_strict": {c: {**v, "cost": fb.cost("gpt-5.4", "batch", usage_by.get("strict:" + c, {}))} for c, v in strict.items()},
           "jobs": {step: {"requests": len(DEMO) if step == "answer" else len(verdicts), "status": "completed",
                           "checks": json.loads(tags.get(f"{step}_checks", "[]")), "retried": int(tags.get("resends", 0))}
                    for step in ("answer", "judge")}}
    DEMO_FILE.write_text(json.dumps(rec, indent=1, ensure_ascii=False))
    print("recording written:", DEMO_FILE)


def attach_verdicts(run_id, verdicts):
    """Each verdict onto its answer's trace as LLM-judge feedback, and the pass rate onto the run."""
    for tr in mlflow.search_traces(run_id=run_id, return_type="list"):
        inputs = json.loads(tr.data.request) if isinstance(tr.data.request, str) else tr.data.request
        v = verdicts.get(f"{inputs['lesson_id']}/{inputs['key']}")
        if v:
            mlflow.log_feedback(trace_id=tr.info.trace_id, name="passes", value=bool(v.get("pass")), rationale=v.get("notes") or None,
                                source=AssessmentSource(source_type="LLM_JUDGE", source_id="gpt-5.4"))
    passed = [bool(v.get("pass")) for v in verdicts.values()]
    mlflow.MlflowClient().log_metric(run_id, "pass_rate", sum(passed) / len(passed) if passed else 0.0)


print("stage now:", follow_up(BATCH_RUN_ID))

**What that cell did.** One check. Live, the job is usually still `validating` or `in_progress` the
first time, so it stops there and you come back later. The replay steps through the real recorded checks
one call at a time. The next cell keeps calling it until the job is judged, which stands in for coming
back later.

In [ ]:
stage = mlflow.get_run(BATCH_RUN_ID).data.tags["stage"]
while stage != "judged":
    if RUN_LIVE:
        time.sleep(60)          # live: better to close the notebook and run 10c again later
    stage = follow_up(BATCH_RUN_ID)
print("stage:", stage)
for step, j in FB_REC["jobs"].items():
    print(f"recorded {step} job: {j['requests']} requests, {j['status']} after {j['minutes']} minutes, {j['retried']} retried")

### 10d. What came back, and how it joins the experiment

Read the result **from MLflow**, the way a teammate would who never ran this notebook.

In [ ]:
rows_out = []
for name, run_id in (("GPT-5.4 mini, inline (Foundry)", inline_run.info.run_id), ("GPT-5.4 mini, batch (Foundry)", BATCH_RUN_ID)):
    r = mlflow.get_run(run_id)
    traces = mlflow.search_traces(run_id=run_id, return_type="list")
    judged = [a for tr in traces for a in tr.info.assessments if a.name == "passes"]
    rows_out.append({"version": name, "answers": len(traces), "judged": len(judged),
                     "pass_rate": r.data.metrics.get("pass_rate"), "cost_per_1000_usd": r.data.metrics.get("cost_per_1000_usd"),
                     "first_words_p95_s": r.data.metrics.get("first_words_p95_s", float("nan")), "stage": r.data.tags.get("stage", "done")})
foundry = pd.DataFrame(rows_out).set_index("version")
foundry.round(4)

**What to look for.**

- **Cost.** The batch costs about half as much per answer as the same model inline, for the same words.
- **Speed.** The batch has no p95 time to first words: it waited in a queue, and that's not what a
  learner feels. The inline sample is where speed comes from.
- **Quality, and a real lesson in it.** The first judge of this run used the prepared answers' checklist
  unchanged, and many answers failed its **standalone** check because they end with a question back to
  the learner. The live tutor's rules allow that; prepared answers mustn't. So that judge didn't fit this
  version. The run was judged again with the check fitted to live answers
  (`fb.judge_prompt(live=True)`, the only change). The next cell compares the two, and logs the
  comparison to MLflow. Before trusting any score, check the judge measures what *this* version is meant
  to do.
- **How sure.** Twenty answers is a demonstration, not a verdict: look how wide a Wilson interval on 20
  is. A real run sends every dev question the plan in section 8 asks for, in one batch per version.
- **The record.** Every answer is a trace with its verdict attached as feedback, inside runs nested
  under the experiment's parent run, next to the practice versions. The same MLflow queries in section 9
  find them all, and the `source` tag keeps the practice runs and the real ones apart.

In [ ]:
judged = []
for name, verdicts in (("strict (prepared answers' checklist)", FB_REC["verdicts_strict"]), ("fitted to live answers", FB_REC["verdicts"])):
    failed = Counter(k for v in verdicts.values() for k in tpe.CHECKS if v.get(k) is False)
    batch_pass = sum(v.get("pass", False) for k, v in verdicts.items() if not k.startswith("inline:"))
    _, lo, hi = tpe.wilson(batch_pass, len(DEMO))
    judged.append({"judge": name, "passed": sum(v.get("pass", False) for v in verdicts.values()), "of": len(verdicts),
                   "batch_pass_rate": batch_pass / len(DEMO), "wilson_low": lo, "wilson_high": hi,
                   **{f"failed_{k}": failed.get(k, 0) for k in tpe.CHECKS}})
judges_table = pd.DataFrame(judged).set_index("judge")
flipped = [c for c in FB_REC["verdicts"] if FB_REC["verdicts"][c].get("pass") != FB_REC["verdicts_strict"].get(c, {}).get("pass")]
print(f"{len(flipped)} verdicts changed when only the standalone check changed")
for c, v in FB_REC["verdicts"].items():
    if not v.get("pass"):
        print(f"- still failing, {c}: {v.get('notes', '')}")

with mlflow.start_run(run_id=parent.info.run_id):          # the comparison belongs to the experiment, logged
    mlflow.log_table(foundry.reset_index().round(4).to_dict(orient="list"), "foundry_inline_vs_batch.json")
    mlflow.log_table(judges_table.reset_index().round(3).to_dict(orient="list"), "foundry_judge_strict_vs_live.json")
    mlflow.log_dict({"flipped": flipped, "judge_prompt_change": fb.LIVE_STANDALONE}, "foundry_judge_change.json")
judges_table.round(3)

The full runner does exactly this for every version. It times an inline sample of each, sends one batch
per version where the model offers batch, sends one judge batch over everything, and then builds the
table from section 6 out of real runs.

## 11. Confirm once, on test

Everything above is **dev**: try as many plans and versions as you like. When the rule has chosen, run
**only the winner and the baseline** on the **test** lessons, once, and report that number. Holdout stays
for a bigger decision, such as switching the live tutor to a different model (the judging notebook,
section 16, shows how to make MLflow enforce this).

The real version of this notebook is one command that runs the versions live, on the Databricks profile or
Foundry resource you choose, after printing the estimate from section 8. It writes a recording with the same columns as `REC`,
so you can load it in place of `simulate()` and run every cell again.

## What you used

| Idea | Course lesson | In this notebook |
|---|---|---|
| Write the rule first | Write the plan before the runs | section 0 |
| Tokens and cost | Tokens and cost | section 1 |
| Cache hit rate | Caching responses | section 2 |
| p95, not the average | Latency that users feel | section 4 |
| Paired comparison | Telling real gains from noise | section 5 |
| The Pareto front | Comparing LLMs for a task | section 6 |
| What the money is worth | Net value per month | section 7 |
| How many questions | How many examples do you need? | section 8 |
| Tracking a run that outlives the session | (beyond the course) | section 10 |

Beyond the course: the provider's prompt cache (its minimum length, its few-minute lifetime, and why
order and timing decide the hit rate), non-inferiority with a lesson-level paired interval, a practice
recording built on real foundations, and batch calls on Foundry: inline for speed, batch for the rest,
and an MLflow run that remembers a job between sessions.